# v0.23.0 — Rich field types

## 1. Intro

SurrealDB has value types that JSON does not: datetimes, durations, decimals, UUIDs, bytes,
ranges. Before v0.23.0 several of them could not be written (a `timedelta`, a `date`, a `set`), a
stored duration could not be read into a `timedelta` field, and `merge()` stored keyword values
such as datetimes as strings. Since v0.23.0 they **round-trip natively**:

* Python annotations — `datetime`, `date`, `timedelta`, `Decimal`, `UUID`, `bytes`, `set[X]` — are
  stored as the server's own types by every write, and read back as the declared type by every read.
* Two ORM types cover what Python has no type for: `Duration`, a duration with SurrealDB's
  nanosecond precision, and `Range[T]`, a range whose bounds are validated as `T`.
* A filter on a rich field compares with the field's type: `trial__gte="7d"` compares with the
  duration `7d`, not with a string. Range membership is `hour__in=Range(...)`.
* `merge()` and `bulk_update()` validate their keyword values against the field, and
  `atomic_increment()` adds a duration to a duration or to a datetime.
* A value the server cannot hold (an integer beyond 64 bits, a decimal beyond its range), or that
  the installed SDK could not read back (a zero duration), is refused before anything is sent.

The ORM behaves the same on SurrealDB 2.x and 3.x: everything below runs top to bottom on either.

In [1]:
import contextlib
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")

# WebSocket: live queries (section 7) need it.
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 2. Models

`Subscription` uses every rich annotation. No import is needed for the Python types; `Duration`,
`Range` and the bound classes come from `surreal_orm_lite` (the bound classes are the SDK's).
`meta: dict[str, Any]` is an untyped field (section 8). `Shift` has a plain `hour: int`, for range
membership in section 5.

The tables are schemaless: `tags` is stored as an array (a `TYPE set<…>` column would hold a native
set, which the surrealdb SDK 2.0.0 cannot read). The notebook starts by dropping its tables, so it
can be re-executed in place; on SurrealDB 3.x removing a missing table raises, hence the suppressed
errors.

In [2]:
from datetime import UTC, date, datetime, timedelta
from decimal import Decimal
from typing import Any
from uuid import UUID

from surreal_orm_lite import (
    BaseSurrealModel,
    BoundExcluded,
    BoundIncluded,
    Duration,
    Range,
    SurrealConfigDict,
)


class Subscription(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="nb_subscription")

    id: str | None = None
    started_at: datetime
    renews_on: date
    trial: timedelta
    grace: Duration | None = None
    price: Decimal
    token: UUID
    active_hours: Range[int]
    period: Range[datetime] | None = None
    logo: bytes | None = None
    tags: set[str] = set()
    meta: dict[str, Any] = {}


class Shift(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="nb_shift")

    id: str | None = None
    hour: int


client = await SurrealDBConnectionManager.get_client()
DEMO_TABLES = ["nb_subscription", "nb_shift"]


async def reset() -> None:
    """Drop every table of this notebook, whether or not it exists."""
    for table in DEMO_TABLES:
        with contextlib.suppress(Exception):
            await client.query(f"REMOVE TABLE {table};", {})


await reset()
print("Demo tables reset")

Demo tables reset


## 3. Save and read back

Each value is stored as the server's own type, and `get()` returns the declared Python type: a
`date` is a `datetime` at midnight UTC on the server and a `date` again here, a `timedelta` is a
SurrealDB `duration`, a `set` an array.

In [3]:
BASIC_TOKEN = UUID("0190c3d1-5c3b-7cc6-8f8a-2d2b1f0e9a11")

await Subscription(
    id="basic",
    started_at=datetime(2026, 1, 2, 9, 30, tzinfo=UTC),
    renews_on=date(2026, 2, 2),
    trial=timedelta(days=7),
    grace=Duration.parse("1h30m1ns"),  # nanoseconds kept
    price=Decimal("9.99"),
    token=BASIC_TOKEN,
    active_hours=Range(BoundIncluded(9), BoundExcluded(17)),  # 9..17, i.e. [9, 17)
    period=Range(BoundIncluded(datetime(2026, 1, 1, tzinfo=UTC)), BoundExcluded(datetime(2027, 1, 1, tzinfo=UTC))),
    logo=b"GIF89a",
    tags={"monthly", "promo"},
    meta={"plan": "basic"},
).save()

basic = await Subscription.objects().get("basic")
for name in Subscription.model_fields:
    value = getattr(basic, name)
    shown = sorted(value) if isinstance(value, set) else value  # a set prints in no fixed order
    print(f"{name:<13} {type(value).__name__:<10} {shown!s}")

id            str        basic
started_at    datetime   2026-01-02 09:30:00+00:00
renews_on     date       2026-02-02
trial         timedelta  7 days, 0:00:00
grace         Duration   1h30m1ns
price         Decimal    9.99
token         UUID       0190c3d1-5c3b-7cc6-8f8a-2d2b1f0e9a11
active_hours  Range      9..17
period        Range      d'2026-01-01T00:00:00Z'..d'2027-01-01T00:00:00Z'
logo          bytes      b'GIF89a'
tags          set        ['monthly', 'promo']
meta          dict       {'plan': 'basic'}


The raw row, read with the SDK's `client.query()`, holds the SDK's types: its `Duration` and
`Range` classes, a `datetime` for both dates, a `list` for the set. The ORM converts them to the
annotation of each field when it builds the model.

In [4]:
def type_name(value: object) -> str:
    """`module.Class` of a value, the module shortened to its package."""
    return f"{type(value).__module__.split('.')[0]}.{type(value).__name__}"


rows = await client.query("SELECT * FROM nb_subscription:basic;", {})
for column, value in sorted(rows[0].items()):
    print(f"{column:<13} {type_name(value)}")

active_hours  surrealdb.Range
grace         surrealdb.Duration
id            surrealdb.RecordID
logo          builtins.bytes
meta          builtins.dict
period        surrealdb.Range
price         decimal.Decimal
renews_on     datetime.datetime
started_at    datetime.datetime
tags          builtins.list
token         uuid.UUID
trial         surrealdb.Duration


## 4. Precision

SurrealDB durations and datetimes have **nanoseconds**; Python's `timedelta` and `datetime` stop at
microseconds. `Duration` keeps the nanoseconds, and `to_timedelta()` floors them.

In [5]:
grace = Duration.parse("1h30m1ns")
print("Duration:      ", grace, "| elapsed =", grace.elapsed, "ns")
print("to_timedelta():", repr(grace.to_timedelta()))
print("compact:       ", grace.to_compact(), "| [seconds, nanoseconds]")

Duration:       1h30m1ns | elapsed = 5400000000001 ns
to_timedelta(): datetime.timedelta(seconds=5400)
compact:        [5400, 1] | [seconds, nanoseconds]


A record written by the server with nanoseconds, here a `d'…05.123456789Z'` literal (deterministic,
unlike `time::now()`): the `Duration` field keeps the nanosecond, the `timedelta` and `datetime`
fields read back truncated to the microsecond.

In [6]:
await Subscription(
    id="precise",
    started_at=datetime(2026, 1, 2, tzinfo=UTC),
    renews_on=date(2026, 2, 2),
    trial=timedelta(days=1),
    price=Decimal("1"),
    token=UUID("0190c3d1-5c3b-7cc6-8f8a-2d2b1f0e9a12"),
    active_hours=Range(BoundIncluded(0), BoundExcluded(24)),
).save()
await client.query(
    "UPDATE nb_subscription:precise SET started_at = d'2026-01-02T03:04:05.123456789Z', trial = 1h30m1ns, grace = 1h30m1ns;",
    {},
)

precise = await Subscription.objects().get("precise")
print("started_at:", precise.started_at.isoformat(), "(server: …05.123456789)")
print("trial:     ", repr(precise.trial), "(timedelta: the nanosecond is lost)")
print("grace:     ", precise.grace, "(Duration: the nanosecond is kept)")

started_at: 2026-01-02T03:04:05.123456+00:00 (server: …05.123456789)
trial:      datetime.timedelta(seconds=5400) (timedelta: the nanosecond is lost)
grace:      1h30m1ns (Duration: the nanosecond is kept)


The truncated datetime is not the stored one, so an `exact` filter with it misses the record.
Filter on a window instead: `gte` the value read, `lt` one microsecond later.

In [7]:
read = precise.started_at
exact = await Subscription.objects().filter(started_at=read).exec()
window = await Subscription.objects().filter(started_at__gte=read, started_at__lt=read + timedelta(microseconds=1)).exec()
print("filter(started_at=read):            ", [s.id for s in exact])
print("filter(started_at__gte=read, __lt=…):", [s.id for s in window])

filter(started_at=read):             []
filter(started_at__gte=read, __lt=…): ['precise']


A SurrealDB decimal holds at most 28 fractional digits. A longer `Decimal` is rounded client-side
exactly as the server rounds it (once, half away from zero), so a filter with the same value
matches what is stored.

`merge()` writes only `price`, so the nanoseconds of `started_at` stay on the server; a full
`update()` of the loaded record would write the truncated value back.

In [8]:
third = Decimal("0.05") / 3
await precise.merge(price=third)
print("Decimal('0.05') / 3:", third)
print("read back:          ", (await Subscription.objects().get("precise")).price)
print("filter(price=third):", [s.id for s in await Subscription.objects().filter(price=third).exec()])

Decimal('0.05') / 3: 0.01666666666666666666666666667
read back:           0.0166666666666666666666666667
filter(price=third): ['precise']


## 5. Typed lookups

A filter value addressed to a rich field is validated against the field's type, then bound: a
string becomes a duration, a datetime or a UUID, a float a `Decimal`. A third subscription gives the
filters something to tell apart.

In [9]:
await Subscription(
    id="pro",
    started_at=datetime(2025, 12, 15, tzinfo=UTC),
    renews_on=date(2026, 12, 15),
    trial=timedelta(days=14),
    price=Decimal("29.99"),
    token=UUID("0190c3d1-5c3b-7cc6-8f8a-2d2b1f0e9a13"),
    active_hours=Range(BoundIncluded(18), BoundIncluded(23)),  # 18..=23
    period=Range(BoundIncluded(datetime(2025, 12, 15, tzinfo=UTC))),  # open end
    tags={"yearly"},
).save()


async def ids(**lookups: Any) -> list[str]:
    """The ids of the subscriptions matching `lookups`, sorted."""
    return sorted(s.id for s in await Subscription.objects().filter(**lookups).exec())


for label, lookups in [
    ("price__lte=9.99", {"price__lte": 9.99}),
    ("trial__gte='7d'", {"trial__gte": "7d"}),
    ("token='0190c3d1-…-2d2b1f0e9a11'", {"token": str(BASIC_TOKEN)}),
    ("started_at__gte='2026-01-01'", {"started_at__gte": "2026-01-01"}),
    ("active_hours__contains=12", {"active_hours__contains": 12}),
]:
    print(f"{label:<34}", await ids(**lookups))

price__lte=9.99                    ['basic', 'precise']
trial__gte='7d'                    ['basic', 'pro']
token='0190c3d1-…-2d2b1f0e9a11'    ['basic']
started_at__gte='2026-01-01'       ['basic', 'precise']
active_hours__contains=12          ['basic', 'precise']


`__in` with a `Range` is range membership on any field, here `hour IN 9..17` on `Shift`. A filter
value that does not fit the field raises `ValueError` before anything is sent.

In [10]:
for hour in (8, 9, 12, 16, 17, 20):
    await Shift(id=f"h{hour}", hour=hour).save()

office = Range(BoundIncluded(9), BoundExcluded(17))
shifts = await Shift.objects().filter(hour__in=office).order_by("hour").exec()
print(f"hour__in={office}:", [s.hour for s in shifts])

try:
    await Subscription.objects().filter(started_at__gt="yesterday").exec()
except ValueError as exc:
    print("ValueError:", exc)

hour__in=9..17: [9, 12, 16]
ValueError: filter(started_at__gt=…): 'yesterday' is not a valid datetime (Input should be a valid datetime or date, input is too short)


## 6. Keyword writes and atomic operations

`merge()` validates each keyword value against the field: `merge(trial="14d")` stores a duration
(before v0.23.0 it stored the string `"14d"`) and sets `trial` to a `timedelta` on the instance.

In [11]:
await basic.merge(trial="14d")
print("instance:", repr(basic.trial))
rows = await client.query("SELECT VALUE trial FROM nb_subscription:basic;", {})
print("stored:  ", type_name(rows[0]))

instance: datetime.timedelta(days=14)
stored:   surrealdb.Duration


`atomic_increment()` takes a duration too, and its statement follows the field's annotation, so it
gives the same result on 2.x (which ignores `+=` with a duration) and 3.x: a duration field gets
`trial = (trial ?? 0ns) + $amount`, a datetime field `started_at = started_at + $amount`. The
amount must be positive.

In [12]:
await basic.atomic_increment("trial", timedelta(days=7))
print("trial:     ", repr(basic.trial))
await basic.atomic_increment("started_at", timedelta(days=1))
print("started_at:", basic.started_at.isoformat())

try:
    await basic.atomic_increment("trial", -timedelta(days=1))
except ValueError as exc:
    print("ValueError:", exc)

trial:      datetime.timedelta(days=21)
started_at: 2026-01-03T09:30:00+00:00
ValueError: atomic_increment(trial): a duration amount must be positive, got datetime.timedelta(days=-1) — SurrealDB cannot store a zero or negative duration; set the new value with merge() instead.


## 7. Live query

A live query cannot carry bound parameters on SurrealDB 2.x, so the ORM writes each filter value
into the statement as a SurrealQL literal: `trial >= 1w`, not a string. One subscription outside the
filter is saved, then one inside: only the second is reported, as a typed `Subscription`.
`asyncio.timeout` keeps the notebook from hanging on a quiet stream.

In [13]:
import asyncio


def new_subscription(sub_id: str, trial: timedelta, price: Decimal = Decimal("4.99")) -> Subscription:
    """A subscription with `trial` and `price`, the other fields fixed."""
    return Subscription(
        id=sub_id,
        started_at=datetime(2026, 3, 1, tzinfo=UTC),
        renews_on=date(2026, 4, 1),
        trial=trial,
        price=price,
        token=UUID("0190c3d1-5c3b-7cc6-8f8a-2d2b1f0e9a14"),
        active_hours=Range(BoundIncluded(6), BoundExcluded(12)),
    )


async with asyncio.timeout(15):
    async with Subscription.objects().filter(trial__gte=timedelta(days=7)).live() as stream:
        await new_subscription("short", timedelta(days=1)).save()  # outside the filter
        await new_subscription("long", timedelta(days=30)).save()
        event = await anext(stream)

print(event.action, event.record_id)
print("instance:", type(event.instance).__name__, "| trial =", repr(event.instance.trial))

CREATE nb_subscription:long
instance: Subscription | trial = datetime.timedelta(days=30)


## 8. Untyped fields keep SurrealDB values

`meta: dict[str, Any]` holds whatever the server sent: here a duration and a range written by
SurrealQL, which load as the SDK's `Duration` and `Range`. Before v0.23.0, `update()` of the loaded
record flattened them into dicts (`{"elapsed": …}`); they are now written back as the duration and
the range they were.

In [14]:
await client.query("UPDATE nb_subscription:basic SET meta = {ttl: 1h30m1ns, window: 1..5};", {})

loaded = await Subscription.objects().get("basic")
print("loaded: ", {key: type_name(value) for key, value in sorted(loaded.meta.items())})
loaded.price = Decimal("8.99")
await loaded.update()  # a full replace, meta included

rows = await client.query("SELECT VALUE meta FROM nb_subscription:basic;", {})
meta = rows[0]
print("stored: ", {key: type_name(value) for key, value in sorted(meta.items())})
print("ttl:   ", repr(meta["ttl"]))
print("window:", repr(meta["window"]))

loaded:  {'ttl': 'surrealdb.Duration', 'window': 'surrealdb.Range'}
stored:  {'ttl': 'surrealdb.Duration', 'window': 'surrealdb.Range'}
ttl:    Duration(elapsed=5400000000001)
window: Range(begin=BoundIncluded(value=1), end=BoundExcluded(value=5))


## 9. JSON

`model_dump_json()` writes a `Duration` and a `Range` as their SurrealQL text (`"1h30m1ns"`,
`"9..17"`), and `model_validate_json()` reads that text back. The other types use Pydantic's
usual JSON forms (a `timedelta` is ISO 8601).

In [15]:
import json

sample = await Subscription.objects().get("pro")
sample.grace = Duration.parse("1h30m1ns")
dumped = sample.model_dump_json()
print(json.dumps(json.loads(dumped), indent=1))

again = Subscription.model_validate_json(dumped)
print("round trip equal:", again == sample)

{
 "id": "pro",
 "started_at": "2025-12-15T00:00:00Z",
 "renews_on": "2026-12-15",
 "trial": "P14D",
 "grace": "1h30m1ns",
 "price": "29.99",
 "token": "0190c3d1-5c3b-7cc6-8f8a-2d2b1f0e9a13",
 "active_hours": "18..=23",
 "period": "d'2025-12-15T00:00:00Z'..",
 "logo": null,
 "tags": [
  "yearly"
 ],
 "meta": {}
}
round trip equal: True


## 10. Guards

Values SurrealDB cannot hold, or that the installed SDK could not read back, are refused with a
`ValueError` before anything is sent, so the connection and its live streams stay up:

* a **zero duration**: the surrealdb SDK 2.0.0 cannot read one back (on a WebSocket the connection
  would be lost), so the ORM's writes refuse to store it; the guard lifts by itself with an SDK
  release that decodes zero durations;
* an **integer beyond 64 bits**, which SurrealDB wraps or rejects;
* a **decimal beyond the server's range** (a 96-bit coefficient).

In [16]:
before = await Subscription.objects().count()
try:
    await new_subscription("zero", timedelta(0)).save()
    print("zero timedelta: stored (the installed surrealdb SDK reads zero durations back)")
except ValueError as exc:
    print("zero timedelta:", exc)

try:
    await Shift(id="huge", hour=2**70).save()
except ValueError as exc:
    print("2**70:", exc)

try:
    await new_subscription("rich", timedelta(days=1), price=Decimal("79228162514264337593543950336")).save()
except ValueError as exc:
    print("Decimal:", exc)

print("subscriptions before:", before, "| after:", await Subscription.objects().count())
print("the connection still answers:", await client.query("RETURN 1;", {}))

zero timedelta: a zero duration cannot be written: the installed surrealdb SDK cannot read a zero duration back (`0ns`): on a WebSocket the connection would be lost. Store `None` instead.
2**70: SurrealDB integers are signed 64-bit: 1180591620717411303424 is outside [-2**63, 2**63 - 1]; use Decimal for larger numbers
Decimal: 79228162514264337593543950336 does not fit a SurrealDB decimal: its integer part exceeds 2**96 - 1 (79228162514264337593543950335) once rounded
subscriptions before: 5 | after: 5
the connection still answers: 1


## 11. Cleanup

Drop the demo tables and close the connection.

In [17]:
await reset()
await SurrealDBConnectionManager.close_connection()
print("Done")

Done
